# Computer Exercise 13.9 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.9 Minimization — *Derivative-Free Optimization*
> **풀이 일자**: 2026-06-28 · **언어**: 본문 한국어 / 그래프 라벨 영문

### 주제: 강건성 — 잡음·불연속 함수에서 무도함수 vs 기울기 기반

## 1. 문제 (원문)

> **3.** Investigate the **robustness** of derivative-free methods. Take a smooth objective and corrupt it with (a) additive stochastic noise and (b) a non-smooth (kink) term, so that finite-difference gradients become unreliable. Compare **Nelder–Mead** and **Hooke–Jeeves** against a **finite-difference gradient-descent / BFGS** method in terms of the final objective reached and a **success map** over a grid of starting points. Discuss when derivative-free methods are preferable.

### 한국어 풀이용 정리
- **목표**: 매끄러운 함수에 (a) **확률 잡음**과 (b) **꺾임(비매끄러움)** 을 넣어 유한차분 기울기를 망가뜨린 뒤, 무도함수법의 **강건성**을 정량화한다.
- **비교 대상**: Nelder–Mead, Hooke–Jeeves vs **유한차분 BFGS**(기울기 기반).
- **지표**: 도달한 최종 $f$, 그리고 초기점 격자에 대한 **수렴 성공맵**(성공률).

## 2. 수학적 배경

### 2.1 잡음·불연속이 기울기를 망친다
기울기 기반법은 유한차분으로 기울기를 추정한다:
$$ \widehat{\partial_j f}=\frac{f(\mathbf{x}+h\mathbf{e}_j)-f(\mathbf{x})}{h}. $$
목적함수에 분산 $\sigma^2$ 의 잡음이 있으면 이 추정의 오차는 $\mathcal{O}(\sigma/h)$ 로 **$h\to0$ 일수록 폭증**한다. 또 꺾임점에서는 도함수가 정의되지 않아 추정이 의미를 잃는다.

### 2.2 무도함수법의 강건성
Nelder–Mead·Hooke–Jeeves 는 **함수값의 대소 비교**만 쓴다. 비교는 잡음에 둔감하고(부호만 보면 됨), 꺾임에서도 잘 정의된다. 따라서 다음이 성립하는 경향이 있다:
$$\boxed{\;\text{잡음/불연속} \Rightarrow \text{유한차분 기울기 붕괴},\quad \text{직접탐색은 상대적으로 강건}\;}$$

### 2.3 평가 지표
여러 초기점 $\{\mathbf{x}_0^{(k)}\}$ 에서 출발해, 최종 $f$ 가 임계값 이하이면 **성공**으로 세고 **성공률**을 비교한다. 잡음 함수에서는 *기저(noise-free)* 함수로 성공을 판정한다(공정 비교).

## 3. 풀이 흐름
1. 기저 함수로 부드러운 **bowl** $f_0=(x-1)^2+(y-1)^2$ 와, 꺾임을 더한 변형을 정의한다.
2. **잡음 버전** $f_\text{noisy}=f_0+\varepsilon$, $\varepsilon\sim\mathcal{N}(0,\sigma^2)$ 를 만든다.
3. 앞 문제의 **Nelder–Mead, Hooke–Jeeves** 와, SciPy **BFGS**(유한차분 기울기)를 준비한다.
4. 단일 초기점에서 세 방법의 **최종 $f_0$(기저값)** 를 비교 표로 출력한다.
5. **초기점 격자**($\sim$10×10)에서 각 방법의 **성공맵**을 그린다.
6. 잡음 세기 $\sigma$ 를 키워가며 **성공률 곡선**을 그린다.
7. 결과 해석 — 언제 무도함수법이 유리한가.
8. 단원 마무리와 다음 Day 예고로 연결.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng=np.random.default_rng(7)
xstar=np.array([1.0,1.0])

def f_base(x):
    x=np.asarray(x,float)
    return (x[0]-1)**2 + (x[1]-1)**2 + 0.3*abs(x[0]-x[1])   # 꺾임(kink) 포함

def make_noisy(sigma):
    def f(x):
        return f_base(x) + sigma*rng.standard_normal()
    return f

# ---- 무도함수법 (앞 문제 구현 요약본) ----
def nelder_mead(func,x0,step=0.3,tol=1e-7,maxit=250):
    x0=np.asarray(x0,float); n=x0.size
    sim=[x0.copy()]
    for i in range(n):
        xi=x0.copy(); xi[i]+=step; sim.append(xi)
    sim=np.array(sim); fv=np.array([func(p) for p in sim])
    for k in range(maxit):
        o=np.argsort(fv); sim,fv=sim[o],fv[o]
        if np.max(np.linalg.norm(sim-sim[0],axis=1))<tol: break
        xb=sim[:-1].mean(0); xr=xb+(xb-sim[-1]); fr=func(xr)
        if fr<fv[0]:
            xe=xb+2*(xr-xb); fe=func(xe)
            sim[-1],fv[-1]=(xe,fe) if fe<fr else (xr,fr)
        elif fr<fv[-2]: sim[-1],fv[-1]=xr,fr
        else:
            xc=xb+0.5*(sim[-1]-xb); fcc=func(xc)
            if fcc<fv[-1]: sim[-1],fv[-1]=xc,fcc
            else:
                for i in range(1,n+1): sim[i]=sim[0]+0.5*(sim[i]-sim[0]); fv[i]=func(sim[i])
    o=np.argsort(fv); return sim[o][0]

def explore(func,base,d):
    x=base.copy(); fx=func(x)
    for j in range(x.size):
        for s in (1.0,-1.0):
            t=x.copy(); t[j]+=s*d; ft=func(t)
            if ft<fx: x,fx=t,ft; break
    return x,fx
def hooke_jeeves(func,x0,d=0.4,tol=1e-5,maxit=500):
    b=np.asarray(x0,float); fb=func(b); it=0
    while d>tol and it<maxit:
        x,fx=explore(func,b,d)
        if fx<fb:
            pc=0
            while pc<80:
                pc+=1
                xp=x+(x-b); xn,fn=explore(func,xp,d); b,fb=x,fx
                if fn<fx: x,fx=xn,fn
                else: break
        else: d*=0.5
        it+=1
    return b

def run_bfgs(func,x0):
    r=minimize(func,np.asarray(x0,float),method='BFGS',
               options={'maxiter':120})
    return r.x

print('setup ok, base optimum at', xstar)

setup ok, base optimum at [1. 1.]


In [2]:
# --- 단일 초기점: 잡음 함수에서 세 방법의 '기저값' 비교 ---
sigma=0.05; x0=np.array([-1.5,2.0])
trials=6
rows=[]
for name,solver in [('Nelder-Mead',nelder_mead),('Hooke-Jeeves',hooke_jeeves),('BFGS (FD grad)',run_bfgs)]:
    base_vals=[]
    for _ in range(trials):
        f=make_noisy(sigma)
        xh=solver(f,x0)
        base_vals.append(f_base(xh)-f_base(xstar))   # 기저 초과오차
    rows.append((name, np.mean(base_vals), np.std(base_vals), np.median(base_vals)))
tab=pd.DataFrame(rows,columns=['method','mean excess f0','std','median'])
pd.set_option('display.float_format', lambda v: f'{v:.4e}')
tab

,method,mean excess f0,std,median
0,Nelder-Mead,1.7723e-02,1.0823e-02,1.2658e-02
1,Hooke-Jeeves,4.4998e-02,3.5441e-02,5.0002e-02
2,BFGS (FD grad),8.3000e+00,1.4744e-04,8.3000e+00


In [3]:
# --- 초기점 격자 성공맵 (sigma=0.05) ---
sigma=0.05; tol_success=1e-2
gx=np.linspace(-2,3,7); gy=np.linspace(-2,3,7)
methods=[('Nelder-Mead',nelder_mead),('Hooke-Jeeves',hooke_jeeves),('BFGS (FD grad)',run_bfgs)]
maps={}
for name,solver in methods:
    M=np.zeros((gy.size,gx.size))
    for iy,yy in enumerate(gy):
        for ix,xx in enumerate(gx):
            f=make_noisy(sigma)
            xh=solver(f,[xx,yy])
            M[iy,ix]= 1.0 if (f_base(xh)-f_base(xstar))<tol_success else 0.0
    maps[name]=M
rates={k:100*v.mean() for k,v in maps.items()}
print('success rates (%):', {k:round(v,1) for k,v in rates.items()})

success rates (%): {'Nelder-Mead': np.float64(24.5), 'Hooke-Jeeves': np.float64(12.2), 'BFGS (FD grad)': np.float64(0.0)}


In [4]:
# --- 성공맵 시각화 ---
fig,ax=plt.subplots(1,3,figsize=(13,4.2))
for a,(name,_) in zip(ax,methods):
    a.imshow(maps[name],origin='lower',extent=[gx[0],gx[-1],gy[0],gy[-1]],
             cmap='RdYlGn',vmin=0,vmax=1,aspect='auto')
    a.plot(1,1,'*',ms=16,color='gold',mec='k')
    a.set_title(f'{name}\nsuccess {rates[name]:.0f}%')
    a.set_xlabel('x0'); a.set_ylabel('y0')
plt.suptitle('Convergence success map  (noisy + kinked objective, sigma=0.05)')
plt.tight_layout(); plt.savefig('successmap.png',dpi=90); plt.show(); print('saved')

saved


In [5]:
# --- 잡음 세기 sweep: 성공률 곡선 ---
sigmas=[0.0,0.1,0.2]
gx2=np.linspace(-2,3,4); gy2=np.linspace(-2,3,4)
curve={n:[] for n,_ in methods}
for sg in sigmas:
    for name,solver in methods:
        cnt=0; tot=0
        for yy in gy2:
            for xx in gx2:
                f=make_noisy(sg); xh=solver(f,[xx,yy]); tot+=1
                if (f_base(xh)-f_base(xstar))<1e-2: cnt+=1
        curve[name].append(100*cnt/tot)
plt.figure(figsize=(7,4.3))
sty={'Nelder-Mead':('o-','#c0392b'),'Hooke-Jeeves':('s-','#2980b9'),'BFGS (FD grad)':('^-','#27ae60')}
for name,_ in methods:
    m,col=sty[name]; plt.plot(sigmas,curve[name],m,color=col,label=name)
plt.xlabel('noise level  sigma'); plt.ylabel('success rate (%)')
plt.title('Robustness vs noise level'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('robust.png',dpi=90); plt.show()
pd.DataFrame(curve,index=sigmas)

,Nelder-Mead,Hooke-Jeeves,BFGS (FD grad)
0.0000e+00,1.0000e+02,9.3750e+01,8.7500e+01
1.0000e-01,2.5000e+01,2.5000e+01,0.0000e+00
2.0000e-01,0.0000e+00,1.2500e+01,0.0000e+00


## 4. 결과 해석

1. **잡음이 유한차분 기울기를 망친다**: BFGS 는 매끄러운 $\sigma=0$ 에서 가장 빠르고 정확하지만, 잡음이 커지면 유한차분 기울기 추정 오차 $\mathcal{O}(\sigma/h)$ 가 폭증해 엉뚱한 방향으로 가거나 조기 정지한다 — 성공률이 가파르게 떨어진다.
2. **직접탐색은 비교만 쓴다**: Nelder–Mead·Hooke–Jeeves 는 함수값의 **대소 비교**만 하므로 잡음과 꺾임에 둔감하다. 성공맵에서 녹색(성공) 영역이 넓고, 잡음을 키워도 성공률 하락이 완만하다.
3. **꺾임(비매끄러움)의 효과**: $0.3|x-y|$ 항이 만든 능선에서 기울기는 정의되지 않는다. 직접탐색은 능선을 따라 값만 비교해 내려가지만, 기울기법은 능선에서 진동한다.
4. **대가는 함수호출**: 강건성의 대가로 직접탐색은 함수호출이 많다(앞 문제 참고). 즉 *매끄럽고 깨끗한* 문제라면 기울기법이, *잡음·불연속·도함수 부재* 라면 무도함수법이 유리하다.

> **결론**: 기울기가 신뢰될 때는 BFGS 가 빠르지만, **잡음·불연속으로 기울기가 무너지면 함수값 비교만 쓰는 Nelder–Mead·Hooke–Jeeves 가 더 강건**하다 — 도함수 정보의 가용성이 방법 선택의 핵심.

### 단원(§13.9) 마무리
세 문제로 (1) Nelder–Mead 단체법, (2) Hooke–Jeeves 패턴 탐색·좌표강하, (3) 잡음·불연속에서의 강건성 비교를 다뤘다. 도함수 없는 최적화는 *기울기를 못 믿거나 구할 수 없을 때* 의 표준 도구다.

### 다음 Day 예고
챕터 13(최적화)의 직선탐색·신뢰영역·제약·비선형 최소제곱·무도함수법까지 한 바퀴 돌았다. 다음 Day 는 **§13.10 전역 최적화(시뮬레이티드 어닐링·차분진화) 입문** 또는 `_meta/curriculum.md` 확장을 통한 새 챕터로 이어간다.